<a href="https://colab.research.google.com/github/williamjacksonuctest/MSAI-699-Capstone/blob/main/W_Jackson_Capstone_Project_Week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets transformers accelerate torchvision scikit-learn

In [ ]:
"""
Capstone Project - Week 3 Baseline Framework (Fixed Dataset Distribution)
Topic: Securing and Verifying Professional Credentials
Author: William Jackson
Dataset: dvgodoy/rvl_cdip_mini (Public HF RVL-CDIP Subset)
"""

import os
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# Ensure reproducibility across Colab runs
np.random.seed(42)
torch.manual_seed(42)

def load_huggingface_data():
    print("[INFO] Loading and shuffling dataset from Hugging Face (dvgodoy/rvl_cdip_mini)...")
    # Load and shuffle the train split using a set seed for reproducible results
    dataset = load_dataset("dvgodoy/rvl_cdip_mini", split="train")
    shuffled_dataset = dataset.shuffle(seed=42)

    texts = []
    images = []
    labels = []

    # Grab a sample cohort of 200 shuffled documents to keep baseline execution quick
    for i, item in enumerate(shuffled_dataset):
        if i >= 200:
            break

        # 'ocr_paragraphs' is a list of strings. We join them to reconstruct the full document text.
        ocr_list = item.get('ocr_paragraphs', [])
        full_text = " ".join(ocr_list) if isinstance(ocr_list, list) else str(ocr_list)

        texts.append(full_text)
        images.append(item['image'].convert('RGB'))
        labels.append(item['label'])

    return texts, images, np.array(labels)

def run_text_baseline(X_train_txt, X_test_txt, y_train, y_test):
    print("\n" + "="*60)
    print("RUNNING MODALITY 1: TEXT-ONLY BASELINE (TF-IDF + LOGISTIC REGRESSION)")
    print("="*60)

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=1000, stop_words='english')
    X_train_vec = vectorizer.fit_transform(X_train_txt)
    X_test_vec = vectorizer.transform(X_test_txt)

    classifier = LogisticRegression(max_iter=1000)
    classifier.fit(X_train_vec, y_train)

    preds = classifier.predict(X_test_vec)
    print(f"[RESULT] Text Baseline Test Accuracy: {accuracy_score(y_test, preds) * 100:.2f}%")
    print("\nDetailed Text Classification Report:")
    print(classification_report(y_test, preds, zero_division=0))

def run_vision_baseline(X_train_img, X_test_img, y_train, y_test):
    print("\n" + "="*60)
    print("RUNNING MODALITY 2: VISION-ONLY BASELINE (RESNET-18 FEATURES + LOGISTIC REGRESSION)")
    print("="*60)

    # Check if Colab GPU acceleration is available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] Processing Vision Pipeline utilizing hardware device: {device}")

    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    # Load pre-trained ResNet Weights
    weights = models.ResNet18_Weights.DEFAULT
    resnet = models.resnet18(weights=weights).to(device)
    resnet.fc = torch.nn.Identity()  # Strip top classifier layer to isolate features
    resnet.eval()

    def extract_features(img_list):
        features = []
        with torch.no_grad():
            for img in img_list:
                tensor = preprocess(img).unsqueeze(0).to(device)
                feat = resnet(tensor).flatten().cpu().numpy()
                features.append(feat)
        return np.array(features)

    X_train_feats = extract_features(X_train_img)
    X_test_feats = extract_features(X_test_img)

    classifier = LogisticRegression(max_iter=1000)
    classifier.fit(X_train_feats, y_train)

    preds = classifier.predict(X_test_feats)
    print(f"[RESULT] Vision Baseline Test Accuracy: {accuracy_score(y_test, preds) * 100:.2f}%")
    print("\nDetailed Vision Classification Report:")
    print(classification_report(y_test, preds, zero_division=0))

if __name__ == "__main__":
    # Ingest directly from Hugging Face Hub
    texts, images, labels = load_huggingface_data()
    print(f"[INFO] Successfully loaded {len(labels)} sample arrays from the public rvl_cdip_mini.")

    # Show the class distribution to verify multiple classes are loaded
    unique_classes, counts = np.unique(labels, return_counts=True)
    print(f"[DEBUG] Label Distribution: {dict(zip(unique_classes, counts))}")

    # Stratified 80/20 split indexing
    split_idx = int(len(labels) * 0.8)

    X_train_txt, X_test_txt = texts[:split_idx], texts[split_idx:]
    X_train_img, X_test_img = images[:split_idx], images[split_idx:]
    y_train, y_test = labels[:split_idx], labels[split_idx:]

    # Run comparative baseline tracks
    run_text_baseline(X_train_txt, X_test_txt, y_train, y_test)
    run_vision_baseline(X_train_img, X_test_img, y_train, y_test)

[INFO] Loading and shuffling dataset from Hugging Face (dvgodoy/rvl_cdip_mini)...
[INFO] Successfully loaded 200 sample arrays from the public rvl_cdip_mini.
[DEBUG] Label Distribution: {np.int64(0): np.int64(9), np.int64(1): np.int64(5), np.int64(2): np.int64(15), np.int64(3): np.int64(16), np.int64(4): np.int64(18), np.int64(5): np.int64(10), np.int64(6): np.int64(10), np.int64(7): np.int64(13), np.int64(8): np.int64(12), np.int64(9): np.int64(12), np.int64(10): np.int64(12), np.int64(11): np.int64(15), np.int64(12): np.int64(12), np.int64(13): np.int64(15), np.int64(14): np.int64(14), np.int64(15): np.int64(12)}

RUNNING MODALITY 1: TEXT-ONLY BASELINE (TF-IDF + LOGISTIC REGRESSION)
[RESULT] Text Baseline Test Accuracy: 37.50%

Detailed Text Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         4
           1       0.00      0.00      0.00         1
           2       1.00      0.33      0.50         3
      

100%|██████████| 44.7M/44.7M [00:00<00:00, 137MB/s]


[RESULT] Vision Baseline Test Accuracy: 40.00%

Detailed Vision Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         4
           1       0.00      0.00      0.00         1
           2       0.25      0.33      0.29         3
           3       1.00      0.67      0.80         3
           4       0.67      0.67      0.67         3
           5       0.00      0.00      0.00         3
           6       0.00      0.00      0.00         0
           7       0.67      0.50      0.57         4
           8       1.00      0.67      0.80         3
           9       0.00      0.00      0.00         1
          10       0.50      0.50      0.50         2
          11       1.00      0.67      0.80         3
          12       0.00      0.00      0.00         1
          13       0.25      1.00      0.40         2
          14       0.50      0.50      0.50         4
          15       0.00      0.00      0.00     

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Structured baseline run metrics
text_data = {
    'Class ID': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    'Document Category': ['Letter', 'Form', 'Email', 'Handwritten', 'Advertisement', 'Scientific Report', 'Scientific Publication', 'Specification', 'File Folder', 'News Article', 'Budget', 'Invoice', 'Presentation', 'Questionnaire', 'Resume', 'Memo'],
    'Precision': [0.00, 0.00, 1.00, 0.25, 0.11, 0.00, 0.00, 1.00, 0.00, 0.00, 0.00, 0.67, 0.00, 0.50, 0.67, 0.00],
    'Recall': [0.00, 0.00, 0.33, 0.33, 0.67, 0.00, 0.00, 0.75, 0.00, 0.00, 0.00, 0.67, 0.00, 1.00, 1.00, 0.00],
    'F1-Score': [0.00, 0.00, 0.50, 0.29, 0.19, 0.00, 0.00, 0.86, 0.00, 0.00, 0.00, 0.67, 0.00, 0.67, 0.80, 0.00],
    'Support': [4, 1, 3, 3, 3, 3, 0, 4, 3, 1, 2, 3, 1, 2, 4, 3]
}

vision_data = {
    'Class ID': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    'Document Category': ['Letter', 'Form', 'Email', 'Handwritten', 'Advertisement', 'Scientific Report', 'Scientific Publication', 'Specification', 'File Folder', 'News Article', 'Budget', 'Invoice', 'Presentation', 'Questionnaire', 'Resume', 'Memo'],
    'Precision': [0.00, 0.00, 0.25, 1.00, 0.67, 0.00, 0.00, 0.67, 1.00, 0.00, 0.50, 1.00, 0.00, 0.25, 0.50, 0.00],
    'Recall': [0.00, 0.00, 0.33, 0.67, 0.67, 0.00, 0.00, 0.50, 0.67, 0.00, 0.50, 0.67, 0.00, 1.00, 0.50, 0.00],
    'F1-Score': [0.00, 0.00, 0.29, 0.80, 0.67, 0.00, 0.00, 0.57, 0.80, 0.00, 0.50, 0.80, 0.00, 0.40, 0.50, 0.00],
    'Support': [4, 1, 3, 3, 3, 3, 0, 4, 3, 1, 2, 3, 1, 2, 4, 3]
}

def export_table_pdf(data, title, filename):
    df = pd.DataFrame(data)

    # Standard format: clean white canvas at high dpi for vector scale
    fig, ax = plt.subplots(figsize=(10, 6.5), dpi=300)
    fig.patch.set_facecolor('white')
    ax.axis('off')

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc='center',
        loc='center'
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.1, 1.4)

    # Explicit styling matching standard reports
    for (row, col), cell in table.get_celld().items():
        cell.set_facecolor('white')
        cell.set_edgecolor('#e0e0e0')
        if row == 0:
            cell.set_text_props(weight='bold')
            cell.set_facecolor('#f5f5f5')

    plt.title(title, fontsize=11, weight='bold', pad=15, color='black')
    plt.tight_layout()
    plt.savefig(filename, format="pdf", facecolor=fig.get_facecolor(), bbox_inches='tight')
    plt.close()

# Generate both clean white-background PDFs
export_table_pdf(text_data, "Figure 1: Text Baseline Classification Report (Accuracy: 37.50%)", "text_baseline_report.pdf")
export_table_pdf(vision_data, "Figure 2: Vision Baseline Classification Report (Accuracy: 40.00%)", "vision_baseline_report.pdf")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

# 1. Text Baseline Data with Summary Rows Included
text_data_with_avgs = {
    'Class ID': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, '', 'Avg', 'Avg'],
    'Document Category': [
        'Letter', 'Form', 'Email', 'Handwritten', 'Advertisement', 'Scientific Report',
        'Scientific Publication', 'Specification', 'File Folder', 'News Article', 'Budget',
        'Invoice', 'Presentation', 'Questionnaire', 'Resume', 'Memo',
        'Accuracy', 'Macro Average', 'Weighted Average'
    ],
    'Precision': [0.00, 0.00, 1.00, 0.25, 0.11, 0.00, 0.00, 1.00, 0.00, 0.00, 0.00, 0.67, 0.00, 0.50, 0.67, 0.00, '', 0.28, 0.34],
    'Recall': [0.00, 0.00, 0.33, 0.33, 0.67, 0.00, 0.00, 0.75, 0.00, 0.00, 0.00, 0.67, 0.00, 1.00, 1.00, 0.00, '', 0.32, 0.38],
    'F1-Score': [0.00, 0.00, 0.50, 0.29, 0.19, 0.00, 0.00, 0.86, 0.00, 0.00, 0.00, 0.67, 0.00, 0.67, 0.80, 0.00, 0.38, 0.26, 0.32],
    'Support': [4, 1, 3, 3, 3, 3, 0, 4, 3, 1, 2, 3, 1, 2, 4, 3, 40, 40, 40]
}

# 2. Vision Baseline Data with Summary Rows Included
vision_data_with_avgs = {
    'Class ID': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, '', 'Avg', 'Avg'],
    'Document Category': [
        'Letter', 'Form', 'Email', 'Handwritten', 'Advertisement', 'Scientific Report',
        'Scientific Publication', 'Specification', 'File Folder', 'News Article', 'Budget',
        'Invoice', 'Presentation', 'Questionnaire', 'Resume', 'Memo',
        'Accuracy', 'Macro Average', 'Weighted Average'
    ],
    'Precision': [0.00, 0.00, 0.25, 1.00, 0.67, 0.00, 0.00, 0.67, 1.00, 0.00, 0.50, 1.00, 0.00, 0.25, 0.50, 0.00, '', 0.36, 0.45],
    'Recall': [0.00, 0.00, 0.33, 0.67, 0.67, 0.00, 0.00, 0.50, 0.67, 0.00, 0.50, 0.67, 0.00, 1.00, 0.50, 0.00, '', 0.34, 0.40],
    'F1-Score': [0.00, 0.00, 0.29, 0.80, 0.67, 0.00, 0.00, 0.57, 0.80, 0.00, 0.50, 0.80, 0.00, 0.40, 0.50, 0.00, 0.40, 0.33, 0.40],
    'Support': [4, 1, 3, 3, 3, 3, 0, 4, 3, 1, 2, 3, 1, 2, 4, 3, 40, 40, 40]
}

def export_table_pdf_updated(data, title, filename):
    df = pd.DataFrame(data)

    # Expand vertical size slightly to comfortably fit the new rows
    fig, ax = plt.subplots(figsize=(10, 7.5), dpi=300)
    fig.patch.set_facecolor('white')
    ax.axis('off')

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc='center',
        loc='center'
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.1, 1.4)

    # Render with custom styles (Headers and Summary Averages get distinct fills)
    num_rows = len(df)
    for (row, col), cell in table.get_celld().items():
        cell.set_facecolor('white')
        cell.set_edgecolor('#e0e0e0')

        # 1. Style Header Row
        if row == 0:
            cell.set_text_props(weight='bold')
            cell.set_facecolor('#f5f5f5')

        # 2. Style the bottom Average/Summary rows (Rows 17, 18, 19 in table index)
        elif row in [num_rows - 2, num_rows - 1, num_rows]:
            cell.set_text_props(weight='bold')
            cell.set_facecolor('#fafafa') # Soft grey for averages block

    plt.title(title, fontsize=12, weight='bold', pad=20, color='black')
    plt.tight_layout()
    plt.savefig(filename, format="pdf", facecolor=fig.get_facecolor(), bbox_inches='tight')
    plt.close()

    # Instantly trigger browser download from Google Colab
    files.download(filename)

# Run and download updated figures
export_table_pdf_updated(text_data_with_avgs, "Figure 1: Text Baseline Classification Report (Accuracy: 37.50%)", "text_baseline_report.pdf")
export_table_pdf_updated(vision_data_with_avgs, "Figure 2: Vision Baseline Classification Report (Accuracy: 40.00%)", "vision_baseline_report.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>